<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/12-supervised-learning/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 12: Exercises

Twelve exercises: four Easy, five Medium and three Hard. You'll write the core algorithms of this module yourself (least squares, ridge, the lasso and softmax regression), build the tools for diagnosing models (learning curves, a bias-variance simulation, variance inflation factors), and implement target encoding without its classic leakage.

Several checks compare your code with scikit-learn's own implementation of the same method, so when they pass, you know your version computes exactly the same thing.

**Run the setup cell first.**

Rules:

- NumPy, pandas and scikit-learn. Where an exercise asks you to write an algorithm yourself, don't call scikit-learn's version of it.
- Solutions are in `03_solutions.ipynb`.

In [ ]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math
import warnings

import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_regression_metrics(f):
    _scenarios("regression_metrics", [
        ("a simple example", lambda: f([3, 5, 7, 9], [4, 5, 6, 11]),
         {"mae": 1.0, "rmse": math.sqrt(1.5), "r2": 1 - 6 / 20}, "MAE = mean |e|, RMSE = sqrt(mean e²), R² = 1 - SSE/SST"),
        ("a perfect model", lambda: f([1.0, 2.0, 3.0], [1.0, 2.0, 3.0]), {"mae": 0.0, "rmse": 0.0, "r2": 1.0}, None),
        ("predicting the mean gives R² = 0", lambda: f([1, 2, 3, 6], [3, 3, 3, 3])["r2"], 0.0, None),
        ("worse than the mean gives a negative R²", lambda: f([1, 2, 3], [3, 4, 5])["r2"], -5.0,
         "nothing stops R² going below 0"),
        ("returns floats", lambda: [type(v) for v in f(np.array([1, 2]), np.array([1, 3])).values()], [float, float, float], None),
        ("different lengths raise", lambda: f([1, 2, 3], [1, 2]), Raises(ValueError), None),
        ("empty input raises", lambda: f([], []), Raises(ValueError), None),
        ("constant y_true raises", lambda: f([4, 4, 4], [4, 5, 3]), Raises(ValueError), "R² divides by the spread of y_true"),
    ])


def check_polynomial_features(f):
    _scenarios("polynomial_features", [
        ("the example", lambda: f([1, 2, 3], 3), np.array([[1, 1, 1], [2, 4, 8], [3, 9, 27]], dtype=float),
         "columns x, x², ..., x^degree"),
        ("degree 1 is a single column", lambda: f(np.array([0.5, -2.0]), 1), np.array([[0.5], [-2.0]]), None),
        ("shape (n, degree), no column of ones", lambda: f(np.linspace(0, 1, 7), 4).shape, (7, 4), "start the powers at 1, not 0"),
        ("negative inputs", lambda: f([-2], 3), np.array([[-2.0, 4.0, -8.0]]), None),
        ("returns floats", lambda: f([1, 2], 2).dtype.kind, "f", None),
        ("degree 0 raises", lambda: f([1, 2], 0), Raises(ValueError), None),
    ])


def check_fit_least_squares(f):
    rng = np.random.default_rng(0)
    X = rng.normal(size=(50, 3))
    y = X @ np.array([2.0, -1.0, 0.5]) + 4 + rng.normal(0, 0.1, 50)
    ref = LinearRegression().fit(X, y)
    X_dup = np.column_stack([X[:, 0], X[:, 0], X[:, 1]])

    def duplicate_predictions():
        w, b = f(X_dup, y)
        return bool(np.allclose(X_dup @ w + b, LinearRegression().fit(X_dup, y).predict(X_dup), atol=1e-8))

    X_near = np.column_stack([X[:, 0], X[:, 0] + 1e-9 * rng.normal(size=50), X[:, 1]])

    def near_duplicate_residual():
        w, b = f(X_near, y)
        best = np.linalg.lstsq(np.column_stack([X_near, np.ones(50)]), y, rcond=None)[0]
        best_rmse = np.sqrt(np.mean((np.column_stack([X_near, np.ones(50)]) @ best - y) ** 2))
        return bool(np.sqrt(np.mean((X_near @ w + b - y) ** 2)) <= best_rmse + 1e-7)

    _scenarios("fit_least_squares", [
        ("matches scikit-learn", lambda: (lambda wb: (wb[0], wb[1]))(f(X, y)), (ref.coef_, ref.intercept_), None),
        ("an exact line is recovered", lambda: f([0, 1, 2, 3], [1, 3, 5, 7]), (np.array([2.0]), 1.0),
         "a 1D X is a single feature: reshape it to (n, 1)"),
        ("b is a float", lambda: type(f(X, y)[1]), float, "return float(coefficients[-1])"),
        ("a duplicated column still gives the right predictions", duplicate_predictions, True,
         "use np.linalg.lstsq; the normal equations fail when XᵀX is singular"),
        ("an almost-duplicated column: the fit is as good as the best possible", near_duplicate_residual, True,
         "the normal equations lose accuracy when columns are nearly identical; lstsq doesn't"),
    ])


def check_add_interactions(f):
    df = pd.DataFrame({"income": [1.0, 2.0, 3.0], "rooms": [4.0, 5.0, 6.0], "age": [10, 20, 30]})

    def unchanged():
        before = df.copy()
        f(df, [("income", "rooms")])
        return df.equals(before)

    _scenarios("add_interactions", [
        ("adds one product column", lambda: f(df, [("income", "rooms")])["income*rooms"].tolist(), [4.0, 10.0, 18.0], None),
        ("columns named 'a*b', in order, after the originals", lambda: list(f(df, [("income", "age"), ("rooms", "age")]).columns),
         ["income", "rooms", "age", "income*age", "rooms*age"], None),
        ("the input is not modified", unchanged, True, "work on df.copy()"),
        ("no pairs: same columns", lambda: list(f(df, []).columns), ["income", "rooms", "age"], None),
        ("a missing column raises KeyError", lambda: f(df, [("income", "size")]), Raises(KeyError), None),
    ])


def check_ridge_closed_form(f):
    rng = np.random.default_rng(1)
    X = rng.normal(size=(60, 4))
    y = X @ np.array([1.0, 2.0, -1.0, 0.0]) + 150 + rng.normal(0, 0.5, 60)

    def matches(alpha):
        w, b = f(X, y, alpha)
        ref = Ridge(alpha=alpha).fit(X, y)
        return bool(np.allclose(w, ref.coef_, atol=1e-8) and abs(b - ref.intercept_) < 1e-8)

    def huge():
        w, b = f(X, y, 1e10)
        return (bool(np.allclose(w, 0, atol=1e-6)), round(b, 4) == round(y.mean(), 4))

    def duplicates():
        Xd = np.column_stack([X[:, 0], X[:, 0], X[:, 1]])
        w, b = f(Xd, y, 1.0)
        ref = Ridge(alpha=1.0).fit(Xd, y)
        return bool(np.allclose(w, ref.coef_, atol=1e-8))

    ols = LinearRegression().fit(X, y)
    _scenarios("ridge_closed_form", [
        ("alpha = 0 is ordinary least squares", lambda: f(X, y, 0.0), (ols.coef_, ols.intercept_), None),
        ("matches scikit-learn's Ridge, alpha = 5", lambda: matches(5.0), True, "solve (Xcᵀ Xc + αI) w = Xcᵀ yc on centred data"),
        ("matches scikit-learn's Ridge, alpha = 100", lambda: matches(100.0), True, None),
        ("huge alpha: weights near 0, intercept stays at mean(y)", huge, (True, True),
         "don't penalise the intercept: centre, solve for w, then b = mean(y) - mean(X)·w"),
        ("handles a duplicated column", duplicates, True, None),
    ])


def check_choose_alpha(f):
    X, y = load_diabetes(return_X_y=True)
    cv = KFold(5, shuffle=True, random_state=0)
    alphas = [0.01, 1.0, 10.0, 100.0, 1000.0]
    ref = pd.Series({a: -cross_val_score(make_pipeline(StandardScaler(), Ridge(alpha=a)), X, y, cv=cv,
                                         scoring="neg_root_mean_squared_error").mean() for a in alphas})
    result = {}

    def run():
        if "v" not in result:
            result["v"] = f(X, y, alphas, cv)
        return result["v"]

    _scenarios("choose_alpha", [
        ("RMSE for each alpha matches a reference (scaler inside the pipeline)", lambda: run()[1].round(6).tolist(),
         ref.round(6).tolist(), "make_pipeline(StandardScaler(), Ridge(alpha=a)), and flip the sign of the scores"),
        ("indexed by alpha, in order", lambda: list(run()[1].index), alphas, None),
        ("best alpha is the one with the lowest RMSE", lambda: run()[0], ref.idxmin(), "rmse.idxmin()"),
    ])


def check_learning_curve_points(f):
    rng = np.random.default_rng(2)
    x = rng.uniform(0, 4, 300)
    X = x[:, None]
    y = np.sin(1.5 * x) + rng.normal(0, 0.3, 300)
    cv = KFold(5, shuffle=True, random_state=0)
    sizes = [10, 40, 160, 240]

    ref = {}
    for n in sizes:
        tr_e, va_e = [], []
        for tr, va in cv.split(X):
            sub = tr[:n]
            m = LinearRegression().fit(X[sub], y[sub])
            tr_e.append(np.sqrt(np.mean((m.predict(X[sub]) - y[sub]) ** 2)))
            va_e.append(np.sqrt(np.mean((m.predict(X[va]) - y[va]) ** 2)))
        ref[n] = (np.mean(tr_e), np.mean(va_e))

    out = {}

    def run():
        if "v" not in out:
            out["v"] = f(LinearRegression(), X, y, sizes, cv)
        return out["v"]

    def model_unfitted():
        model = LinearRegression()
        f(model, X, y, [20], cv)
        return hasattr(model, "coef_")

    _scenarios("learning_curve_points", [
        ("columns and index", lambda: (list(run().columns), list(run().index)), (["train_rmse", "val_rmse"], sizes), None),
        ("values match a reference", lambda: [tuple(np.round(run().loc[n].to_numpy(), 6)) for n in sizes],
         [tuple(np.round(ref[n], 6)) for n in sizes], "first n indices of each training fold; average over folds"),
        ("training error rises and validation error falls with more data", lambda: (
            bool(run()["train_rmse"].iloc[0] < run()["train_rmse"].iloc[-1]), bool(run()["val_rmse"].iloc[0] > run()["val_rmse"].iloc[-1])),
         (True, True), None),
        ("the model passed in is not fitted", model_unfitted, False, "fit clone(model), not the model itself"),
    ])


def check_bias_variance(f):
    x_eval = np.linspace(0, 4, 40)

    def truth(x):
        return np.sin(1.5 * x)

    def flat(x):
        return np.full_like(x, 2.0)

    def run(fit_predict, tr=truth, n_train=20, repeats=200, seed=0):
        return f(fit_predict, tr, x_eval, 0.5, n_train, repeats, np.random.default_rng(seed))

    def perfect():
        return tuple(round(v, 12) for v in run(lambda xt, yt, xe: truth(xe)))

    def zero():
        b, v = run(lambda xt, yt, xe: np.zeros_like(xe))
        return (round(b, 10), round(v, 10))

    def mean_model_variance():
        b, v = run(lambda xt, yt, xe: np.full_like(xe, yt.mean()), tr=flat, n_train=25, repeats=4000)
        return 0.8 < v / (0.5 ** 2 / 25) < 1.2 and b < 0.002

    def reproducible():
        fp = lambda xt, yt, xe: np.polynomial.Polynomial.fit(xt, yt, 3)(xe)
        return run(fp, seed=3) == run(fp, seed=3)

    def calls():
        count = []

        def fp(xt, yt, xe):
            count.append((len(xt), len(yt)))
            return np.zeros_like(xe)
        run(fp, n_train=15, repeats=7)
        return count

    _scenarios("bias_variance", [
        ("a model that returns the truth: no bias, no variance", perfect, (0.0, 0.0), None),
        ("always predicting 0: bias² = mean(truth²), no variance", zero, (round(float(np.mean(truth(x_eval) ** 2)), 10), 0.0),
         "bias² compares the average prediction with truth(x_eval)"),
        ("predicting mean(y_train): variance close to noise² / n", mean_model_variance, True,
         "variance across repeats for each point (axis=0), then averaged over the points"),
        ("calls fit_predict n_repeats times with n_train points", calls, [(15, 15)] * 7, None),
        ("same seed, same result", reproducible, True, "use the rng passed in for all the randomness"),
    ])


def check_vif(f):
    X, _ = load_diabetes(return_X_y=True, as_frame=True)

    def independent():
        rng = np.random.default_rng(0)
        data = pd.DataFrame(rng.normal(size=(2000, 3)), columns=["a", "b", "c"])
        return bool(np.all(f(data).between(1.0, 1.02)))

    def known():
        rng = np.random.default_rng(1)
        a = rng.normal(size=5000)
        b = a + rng.normal(size=5000)               # R² of a on b is about 0.5, so VIF about 2
        return bool(np.all(f(pd.DataFrame({"a": a, "b": b})).between(1.85, 2.15)))

    def duplicate():
        data = pd.DataFrame({"a": [1.0, 2.0, 3.0, 5.0], "b": [2.0, 4.0, 6.0, 10.0], "c": [1.0, 0.0, 1.0, 3.0]})
        return bool(np.isinf(f(data)["a"]) and np.isinf(f(data)["b"]))

    _scenarios("vif", [
        ("unrelated features have VIF close to 1", independent, True, None),
        ("two related features: VIF close to the theoretical 2", known, True, "VIF = 1 / (1 - R²), regressing each column on the others"),
        ("diabetes: s1 and s2 are strongly inflated", lambda: bool(f(X)["s1"] > 20 and f(X)["s2"] > 20), True, None),
        ("indexed by column name, in order", lambda: list(f(X).index), list(X.columns), None),
        ("a perfectly redundant feature gets inf", duplicate, True, "return np.inf when R² is 1"),
    ])


def check_lasso_coordinate_descent(f):
    X, y = load_diabetes(return_X_y=True)
    Xs = StandardScaler().fit_transform(X)

    def matches(alpha):
        w, b = f(Xs, y, alpha)
        ref = Lasso(alpha=alpha, tol=1e-12, max_iter=100_000).fit(Xs, y)
        same_zeros = bool(np.array_equal(w == 0, ref.coef_ == 0))
        return (bool(np.allclose(w, ref.coef_, atol=1e-4)), abs(b - ref.intercept_) < 1e-4, same_zeros)

    def huge():
        w, b = f(Xs, y, 1000.0)
        return (bool(np.all(w == 0)), round(b, 6) == round(float(y.mean()), 6))

    def small_is_ols():
        w, _ = f(Xs, y, 1e-6)
        return bool(np.allclose(w, LinearRegression().fit(Xs, y).coef_, atol=1e-2))

    _scenarios("lasso_coordinate_descent", [
        ("matches scikit-learn's Lasso, alpha = 1 (weights, intercept, zeros)", lambda: matches(1.0), (True, True, True),
         "w_j = S(rho_j, alpha) / z_j with rho_j computed from the current residual"),
        ("matches at alpha = 10, with more weights at exactly zero", lambda: matches(10.0), (True, True, True),
         "S(rho, alpha) = sign(rho) * max(|rho| - alpha, 0) gives exact zeros"),
        ("a huge alpha removes every feature; intercept is mean(y)", huge, (True, True), None),
        ("a tiny alpha is close to least squares", small_is_ols, True, None),
    ])


def check_softmax_regression(f):
    rng = np.random.default_rng(4)
    centres = np.array([[0.0, 0.0], [2.5, 0.5], [1.0, 2.5]])
    y = np.repeat([0, 1, 2], 60)
    X = centres[y] + rng.normal(0, 0.8, size=(180, 2))
    C = 1.0

    out = {}

    def run():
        if "v" not in out:
            out["v"] = f(X, y, 3, lr=0.5, epochs=3000, l2=1 / (C * len(y)))
        return out["v"]

    def proba(W, b):
        Z = X @ W + b
        Z = Z - Z.max(axis=1, keepdims=True)
        P = np.exp(Z)
        return P / P.sum(axis=1, keepdims=True)

    def matches_sklearn():
        W, b, _ = run()
        ref = LogisticRegression(C=C, tol=1e-10, max_iter=10_000).fit(X, y)
        return bool(np.abs(proba(W, b) - ref.predict_proba(X)).max() < 2e-3)

    _scenarios("softmax_regression", [
        ("shapes: W (d, k), b (k,), one loss per epoch", lambda: (run()[0].shape, run()[1].shape, len(run()[2])), ((2, 3), (3,), 3000), None),
        ("the first loss is log(3), the uniform guess", lambda: round(float(run()[2][0]), 6), round(math.log(3), 6),
         "record the loss before the first update; start from zeros"),
        ("the loss goes down", lambda: bool(run()[2][-1] < run()[2][0] * 0.5 and np.all(np.diff(run()[2]) <= 1e-12)), True,
         "dZ = (P - Y) / n, dW = Xᵀ dZ + l2·W, db = dZ.sum(axis=0)"),
        ("matches scikit-learn's LogisticRegression with l2 = 1 / (C·n)", matches_sklearn, True,
         "penalise W only, with (l2 / 2)·ΣW² in the loss and l2·W in the gradient"),
        ("classifies the training data well", lambda: float(np.mean(np.argmax(proba(*run()[:2]), axis=1) == y)) > 0.85, True, None),
    ])


def check_target_encode_oof(f):
    cats = np.array(["a", "a", "a", "b", "b", "c", "c", "c", "c", "d"])
    y = np.array([1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0])

    def reference(categories, target, n_splits, smoothing, seed):
        enc = np.empty(len(target))
        for fi, ei in KFold(n_splits, shuffle=True, random_state=seed).split(categories):
            m = target[fi].mean()
            df = pd.DataFrame({"c": categories[fi], "y": target[fi]}).groupby("c")["y"].agg(["sum", "count"])
            mp = ((df["sum"] + smoothing * m) / (df["count"] + smoothing)).to_dict()
            enc[ei] = [mp.get(c, m) for c in categories[ei]]
        return enc

    def own_target_excluded():
        enc, _, _ = f(cats, y, n_splits=5, smoothing=0.0, seed=0)
        lone = int(np.where(cats == "d")[0][0])
        others = np.ones(len(y), bool)
        for fi, ei in KFold(5, shuffle=True, random_state=0).split(cats):
            if lone in ei:
                return round(enc[lone], 10) == round(float(y[fi].mean()), 10)
        return False

    def leakage():
        rng = np.random.default_rng(0)
        c = rng.integers(0, 1000, 2000).astype(str)          # about two rows per category
        target = rng.normal(size=2000)                     # pure noise: nothing to learn
        enc, _, _ = f(c, target, n_splits=5, smoothing=0.0, seed=1)
        naive = pd.Series(target).groupby(c).transform("mean").to_numpy()
        return (bool(abs(np.corrcoef(enc, target)[0, 1]) < 0.1), bool(np.corrcoef(naive, target)[0, 1] > 0.5))

    def full_mapping():
        _, mapping, m = f(cats, y, n_splits=5, smoothing=2.0, seed=0)
        return ({k: round(v, 6) for k, v in mapping.items()}, round(m, 6))

    gm = y.mean()
    expected_map = {"a": round((2 + 2 * gm) / (3 + 2), 6), "b": round((0 + 2 * gm) / (2 + 2), 6),
                    "c": round((3 + 2 * gm) / (4 + 2), 6), "d": round((1 + 2 * gm) / (1 + 2), 6)}
    _scenarios("target_encode_oof", [
        ("matches a reference, smoothing 2", lambda: np.round(f(cats, y, n_splits=5, smoothing=2.0, seed=0)[0], 10),
         np.round(reference(cats, y, 5, 2.0, 0), 10), "for each fold, compute the smoothed means from the other folds only"),
        ("a category seen once gets the other folds' mean, not its own target", own_target_excluded, True,
         "never include a row's own target in its encoding"),
        ("the mapping for new data uses all rows", full_mapping, (expected_map, round(gm, 6)), None),
        ("on pure noise: out-of-fold shows no signal, in-sample would fake one", leakage, (True, True), None),
        ("lists work too", lambda: len(f(list(cats), list(y))[0]), 10, None),
    ])


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `regression_metrics`

Write `regression_metrics(y_true, y_pred)` that returns a dict with three floats:

- `"mae"`: the mean absolute error
- `"rmse"`: the root mean squared error
- `"r2"`: `1 - (sum of squared errors) / (sum of squared differences between y_true and its mean)`

Accept lists or arrays. Raise `ValueError` if the inputs have different lengths or are empty, and if every value of `y_true` is the same (R² is undefined then, because the denominator is zero).

In [ ]:
def regression_metrics(y_true, y_pred) -> dict:
    # your code here
    pass

In [ ]:
check_regression_metrics(regression_metrics)

### Exercise 2: `polynomial_features`

Write `polynomial_features(x, degree)` that turns a 1D input into polynomial columns:

```
polynomial_features([1, 2, 3], 3)  ->  [[1, 1, 1],
                                        [2, 4, 8],
                                        [3, 9, 27]]
```

- Return a float array of shape `(n, degree)` with columns `x, x², ..., x^degree` (no column of ones).
- `x` may be a list or an array.
- Raise `ValueError` if `degree` is less than 1.

In [ ]:
def polynomial_features(x, degree: int):
    # your code here
    pass

In [ ]:
check_polynomial_features(polynomial_features)

### Exercise 3: `fit_least_squares`

Write `fit_least_squares(X, y)` that fits linear regression with an intercept and returns `(w, b)`: a float array of weights, one per feature, and a float intercept.

- `X` may be 2D `(n, d)` or 1D (a single feature).
- Use `np.linalg.lstsq` on `X` with a column of ones added.

The check includes data where one column is an exact copy of another, and data where one is an almost exact copy. Your function must still give the best possible fit there.

In [ ]:
def fit_least_squares(X, y):
    # your code here
    pass

In [ ]:
check_fit_least_squares(fit_least_squares)

### Exercise 4: `add_interactions`

Write `add_interactions(df, pairs)` that returns a **new** DataFrame with one extra column per pair of column names, holding their product.

```
add_interactions(df, [("income", "rooms")])   # adds a column named "income*rooms"
```

- Name each new column `"a*b"` for the pair `("a", "b")`, in the order given.
- Don't modify `df`.
- Raise `KeyError` if a pair names a column that doesn't exist.

In [ ]:
def add_interactions(df: pd.DataFrame, pairs) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_add_interactions(add_interactions)

## Medium

### Exercise 5: `ridge_closed_form`

Write `ridge_closed_form(X, y, alpha)` that fits ridge regression exactly and returns `(w, b)`.

The loss is `sum((y - X w - b)²) + alpha * sum(w²)`. **The intercept `b` is not penalised.** The standard way to arrange that:

1. Centre `X` (subtract each column's mean) and `y` (subtract its mean).
2. Solve `(Xcᵀ Xc + alpha · I) w = Xcᵀ yc`.
3. Recover `b = mean(y) - mean(X) · w`.

Use `np.linalg.solve`. With `alpha = 0` the answer should match ordinary least squares.

In [ ]:
def ridge_closed_form(X, y, alpha: float):
    # your code here
    pass

In [ ]:
check_ridge_closed_form(ridge_closed_form)

### Exercise 6: `choose_alpha`

Write `choose_alpha(X, y, alphas, cv)` that picks a ridge penalty by cross-validation.

- For each alpha, build `make_pipeline(StandardScaler(), Ridge(alpha=alpha))` and compute its mean cross-validated RMSE with `cross_val_score(..., cv=cv, scoring="neg_root_mean_squared_error")` (the scores are negative, so flip the sign).
- Return `(best_alpha, rmse)`: the alpha with the lowest RMSE, and a `pd.Series` of RMSE values indexed by alpha, in the order given.

The scaler must be **inside** the pipeline.

In [ ]:
def choose_alpha(X, y, alphas, cv):
    # your code here
    pass

In [ ]:
check_choose_alpha(choose_alpha)

### Exercise 7: `learning_curve_points`

Write `learning_curve_points(model, X, y, sizes, cv)` that computes a learning curve by hand.

For each size `n` in `sizes`, and for each split of `cv` (`cv.split(X)`):

1. Take the **first `n`** indices of the split's training indices, in the order the split returns them.
2. Fit a fresh copy of the model on those rows (`sklearn.base.clone(model)`).
3. Compute the RMSE on those same `n` training rows, and on the split's validation rows.

Return a DataFrame indexed by size, with columns `"train_rmse"` and `"val_rmse"`, each averaged over the splits. `X` and `y` are NumPy arrays.

In [ ]:
from sklearn.base import clone


def learning_curve_points(model, X, y, sizes, cv) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_learning_curve_points(learning_curve_points)

### Exercise 8: `bias_variance`

Write `bias_variance(fit_predict, truth, x_eval, noise_sd, n_train, n_repeats, rng)` that estimates bias² and variance by simulation, as in lesson section 5.

Repeat `n_repeats` times:

1. Draw `n_train` inputs uniformly between `x_eval.min()` and `x_eval.max()` with `rng.uniform`.
2. Make targets `truth(x) + rng.normal(0, noise_sd, n_train)`.
3. Call `fit_predict(x_train, y_train, x_eval)`, which fits a model and returns its predictions at `x_eval`.

With the predictions stacked into an array of shape `(n_repeats, len(x_eval))`, return `(bias_sq, variance)` as floats:

- `bias_sq`: the mean over `x_eval` of `(average prediction - truth(x_eval))²`
- `variance`: the mean over `x_eval` of the variance of the predictions across repeats (`ddof=0`)

In [ ]:
def bias_variance(fit_predict, truth, x_eval, noise_sd: float, n_train: int, n_repeats: int, rng):
    # your code here
    pass

In [ ]:
check_bias_variance(bias_variance)

### Exercise 9: `vif`

The **variance inflation factor** measures how much a feature is explained by the others, the multicollinearity problem from lesson section 2. For feature `j`:

1. Fit a linear regression (with intercept) predicting column `j` from **all the other** columns.
2. Compute its R² on the same data.
3. `VIF_j = 1 / (1 - R²_j)`.

Write `vif(X)` for a DataFrame `X` that returns a `pd.Series` of VIF values indexed by column name, in the original column order.

A VIF of 1 means the feature is unrelated to the others; values above about 5 to 10 usually signal trouble for interpreting weights. If a feature is perfectly predicted by the others (R² of 1), return `np.inf` for it.

In [ ]:
def vif(X: pd.DataFrame) -> pd.Series:
    # your code here
    pass

In [ ]:
check_vif(vif)

## Hard

### Exercise 10: `lasso_coordinate_descent`

Write `lasso_coordinate_descent(X, y, alpha, n_iter=1000, tol=1e-10)` that fits the lasso and returns `(w, b)`. The objective (the same as scikit-learn's `Lasso`):

$$\frac{1}{2n} \sum_i (y_i - \mathbf{x}_i \cdot \mathbf{w} - b)^2 + \alpha \sum_j |w_j|$$

**Coordinate descent** updates one weight at a time, holding the others fixed. After centring `X` and `y` (as in ridge, so the intercept drops out), the best value for weight `j` given all the others is:

$$w_j = \frac{S(\rho_j, \alpha)}{z_j}, \quad \rho_j = \frac{1}{n}\sum_i x_{ij}\,(r_i + x_{ij} w_j), \quad z_j = \frac{1}{n}\sum_i x_{ij}^2$$

where `r = yc - Xc @ w` is the current residual and `S(ρ, α) = sign(ρ) · max(|ρ| - α, 0)` is the **soft-thresholding** function. Sweep over all weights repeatedly, keeping `r` up to date, until the largest change in a sweep is below `tol` (or after `n_iter` sweeps). Then `b = mean(y) - mean(X) · w`.

In [ ]:
def lasso_coordinate_descent(X, y, alpha: float, n_iter: int = 1000, tol: float = 1e-10):
    # your code here
    pass

In [ ]:
check_lasso_coordinate_descent(lasso_coordinate_descent)

### Exercise 11: `softmax_regression`

Write `softmax_regression(X, y, n_classes, lr=0.5, epochs=3000, l2=0.0)` that trains multinomial logistic regression with full-batch gradient descent and returns `(W, b, losses)`.

- `X` has shape `(n, d)`; `y` holds integer class labels `0` to `n_classes - 1`. Start from `W = zeros((d, k))` and `b = zeros(k)`.
- Scores `Z = X @ W + b`; probabilities `P = softmax(Z)` row by row (use the stable version from Module 9).
- Loss: mean cross-entropy, plus `(l2 / 2) * sum(W²)` (the bias isn't penalised).
- Gradients: with `Y` the one-hot matrix of labels, `dZ = (P - Y) / n`, so `dW = Xᵀ dZ + l2 · W` and `db = sum of dZ over rows`.
- `losses[i]` is the loss **before** update `i` (a list or array of length `epochs`).

With `l2 = 1 / (C · n)`, your answer should match scikit-learn's `LogisticRegression(C=C)` once training has converged.

In [ ]:
def softmax_regression(X, y, n_classes: int, lr: float = 0.5, epochs: int = 3000, l2: float = 0.0):
    # your code here
    pass

In [ ]:
check_softmax_regression(softmax_regression)

### Exercise 12: `target_encode_oof`

**Target encoding** replaces each category with the average target value for that category. It handles categories with many values (thousands of zip codes, say) where one-hot encoding would create thousands of columns. It's also a classic source of leakage: if a row's own target is included in its category's average, the feature partly **contains the answer**.

Write `target_encode_oof(categories, y, n_splits=5, smoothing=10.0, seed=0)` that encodes **out of fold**, and returns `(encoded, mapping, global_mean)`:

- Split the rows with `KFold(n_splits, shuffle=True, random_state=seed)`. For the rows in each fold, compute the encoding using **only the other folds**:
  `(sum of y in the category + smoothing · m) / (count in the category + smoothing)`, where `m` is the mean of `y` over the other folds. A category absent from the other folds gets `m`.
- `encoded`: a float array, one value per row.
- `mapping` and `global_mean`: the same formula computed on **all** rows (a dict from category to value, and the overall mean of `y`), for encoding new data later.

`categories` and `y` may be lists or arrays.

In [ ]:
from sklearn.model_selection import KFold


def target_encode_oof(categories, y, n_splits: int = 5, smoothing: float = 10.0, seed: int = 0):
    # your code here
    pass

In [ ]:
check_target_encode_oof(target_encode_oof)

## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the mini project in `04_mini_project.ipynb`, where these ideas meet a real dataset: predicting house values from the 1990 California census with linear models, feature engineering and regularisation.